# Station Stacking v6 - KLGA

Wide HRRR/GFS same-day 11am notebook for `KLGA`.

This version uses source-owned v5 feature engineering, additive morning temperature trend features, and durable Optuna SQLite storage. Artifacts are written to `data/calibration/station_stacking_v6`.


In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

STATION_ID = "KLGA"
FAST_MODE = False
OPTUNA_TRIALS = 100
STACK_OPTUNA_TRIALS = 50
OPTUNA_STARTUP_TRIALS = 30
STACK_OPTUNA_STARTUP_TRIALS = 30
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.calibration.station_stacking import (
    StationStackingConfig,
    V6_FEATURE_COLUMNS,
    missing_model_dependencies,
    run_station_year_split_experiment,
)


## V6 Feature Engineering

`feature_version="v6"` applies the source-owned v5 feature block and includes the 11am observation trend columns when present in the current-observation cache.


In [3]:
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]

V6_FEATURE_COLUMNS


['v2_recent_heat_anomaly_f',
 'v2_recent_heat_momentum_f',
 'v2_morning_warmup_to_consensus_f',
 'v2_consensus_minus_7d_actual_f',
 'v2_spread_per_warmup_f',
 'v2_humidity_warmup_interaction',
 'v3_high_so_far_above_current_f',
 'v3_remaining_warmup_from_high_so_far_f',
 'v3_high_so_far_minus_lag_1d_f',
 'v3_high_so_far_minus_7d_actual_f',
 'v3_remaining_warmup_per_spread_f',
 'v3_humidity_remaining_warmup_interaction',
 'v4_forecast_precip_total_mean_mm',
 'v4_forecast_precip_total_max_mm',
 'v4_forecast_precip_total_spread_mm',
 'v4_forecast_precip_max_1h_mean_mm',
 'v4_forecast_precip_hours_mean',
 'v4_forecast_precip_intensity_mean',
 'v4_forecast_precip_intensity_max',
 'v4_any_forecast_precip',
 'v4_all_forecast_precip',
 'v4_observed_precip_any',
 'v4_observed_precip_recent_mm_est',
 'v4_forecast_total_minus_observed_recent_mm',
 'v4_forecast_observed_precip_match',
 'v4_forecast_wet_observed_dry',
 'v4_observed_wet_forecast_dry',
 'v4_precip_humidity_interaction',
 'v4_precip_r

## Model Scores


In [4]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v6",
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v6/KLGA_optuna.sqlite3')

In [5]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-06-10 13:39:45,549] A new study created in RDB with name: KLGA_v6_base_xgboost_mae_f
[I 2026-06-10 13:39:54,977] Trial 0 finished with value: 1.494323586003577 and parameters: {'n_estimators': 799, 'learning_rate': 0.12369619597856178, 'max_depth': 6, 'min_child_weight': 2.385234757844707, 'gamma': 0.7800932022121826, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.6677615511747083}. Best is trial 0 with value: 1.494323586003577.
[I 2026-06-10 13:44:47,801] Trial 1 finished with value: 1.497476649444597 and parameters: {'n_estimators': 1440, 'learning_rate': 0.0032515743808034223, 'max_depth': 8, 'min_child_weight': 8.23143373099555, 'gamma': 1.0616955533913808, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.5917022549267169, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.2922905212920093}. Best is trial 0 with value: 1.494323586003577.
[I 2026-06-10 13:46:26,595] Trial 2 finished with value:

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,487,1.358978,1.774098
1,validation_2024_2025,lightgbm,487,1.416813,1.803390
2,validation_2024_2025,catboost,487,1.392640,1.840188
3,validation_2024_2025,hrrr_raw,487,1.899590,2.583317
4,validation_2024_2025,gfs_raw,487,2.350877,3.124910
5,test_2026,xgboost,51,1.739380,2.142030
6,test_2026,lightgbm,51,1.732321,2.145685
7,test_2026,catboost,51,1.840768,2.406455
8,test_2026,ridge_stack,51,1.698213,2.153980
9,test_2026,hrrr_raw,51,2.600864,3.836077


## Morning Trend Coverage


In [6]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,0.0
1,observed_temp_change_last_3h_f,0.0
2,observed_morning_warmup_rate_f_per_hour,0.0
3,observed_high_so_far_change_since_9am_f,0.0


In [7]:
result.feature_columns.loc[result.feature_columns["feature"].isin(TREND_COLUMNS)]


,feature,kind


## Rounded Within 1F Accuracy


In [8]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="test_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
4,test_2026,ridge_stack,51,29,56.862745
0,test_2026,catboost,51,26,50.980392
5,test_2026,xgboost,51,26,50.980392
2,test_2026,hrrr_raw,51,25,49.019608
3,test_2026,lightgbm,51,25,49.019608
1,test_2026,gfs_raw,51,17,33.333333
10,validation_2024_2025,xgboost,487,313,64.271047
6,validation_2024_2025,catboost,487,308,63.244353
9,validation_2024_2025,lightgbm,487,302,62.012320
8,validation_2024_2025,hrrr_raw,487,252,51.745380


## Version Comparison


In [9]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,ridge_stack,51,1.698213,2.153980,v6
1,test_2026,catboost,51,1.702215,2.173142,v5
2,test_2026,lightgbm,51,1.732321,2.145685,v6
3,test_2026,xgboost,51,1.739380,2.142030,v6
4,test_2026,ridge_stack,51,1.762395,2.269680,v5
5,test_2026,catboost,51,1.840768,2.406455,v6
6,test_2026,xgboost,51,1.970093,2.490840,v5
7,test_2026,lightgbm,51,1.982511,2.443979,v5
8,test_2026,hrrr_raw,51,2.600864,3.836077,v5
9,test_2026,hrrr_raw,51,2.600864,3.836077,v6


## 2026 Weather Brackets


In [10]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,51,1.739380,2.142030,39.215686
1,lightgbm,51,1.732321,2.145685,39.215686
2,catboost,51,1.840768,2.406455,45.098039
3,ridge_stack,51,1.698213,2.153980,39.215686
4,hrrr_raw,51,2.600864,3.836077,33.333333
5,gfs_raw,51,3.161652,4.378131,23.529412
